# Sprint 08 · Linear probing ImageNet · kaggle

Antes de executar: leia `sprints/README.md`. Input obrigatório: manifest.csv e as imagens de desenvolvimento. Não usa o teste final. Não cole tokens no notebook.

In [ ]:
from pathlib import Path
import subprocess, sys
from zipfile import ZipFile

mounted = Path('/kaggle/input')
slug = 'subset-multibase-cancer-mama'
roots = [p for p in (mounted / slug, mounted / 'datasets' / 'marcowandraski' / slug) if p.is_dir()]
DATA = roots[0] if len(roots) == 1 else mounted / slug
zips = sorted(mounted.rglob('mammo-codigo.zip'))
if len(zips) != 1:
    raise RuntimeError('Anexe exatamente um mammo-codigo.zip como input de código')
SOURCE_ZIP = zips[0]
PROJECT = Path('/kaggle/working/mammo-codigo')
OUTPUT = Path('/kaggle/working/resultados')
# Altere DATA se o input estiver em outro caminho. Não aponte para o teste final.
SMOKE = True  # S7–S9: uma época, sem pesos externos. False usa o protocolo configurado.
ARM = 'batch-16-fixed'  # somente S9; um braço por execução

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f'Anexe o ZIP de código em {SOURCE_ZIP}; veja sprints/README.md')
with ZipFile(SOURCE_ZIP) as z:
    for item in z.infolist():
        dest = (PROJECT / item.filename).resolve()
        if not dest.is_relative_to(PROJECT.resolve()):
            raise ValueError('Arquivo de código com caminho inválido')
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
if not (PROJECT / 'configs' / 'sprints.toml').is_file():
    raise RuntimeError('ZIP incorreto: gere com python scripts/package_sprints.py')


In [ ]:
from project_cnn_mammo.training_run import development_manifest
MANIFEST = DATA / 'manifest.csv'
if not MANIFEST.is_file():
    MANIFEST = SOURCE_ZIP.parent / 'manifest.csv'  # sidecar, sem recarregar as imagens
if not MANIFEST.is_file():
    raise FileNotFoundError('Anexe manifest.csv com IDs e split reais; veja sprints/README.md')
entries = development_manifest(DATA, MANIFEST)
print(f'Manifesto válido: {len(entries)} imagens de desenvolvimento')
import torch, torchvision  # use o par já instalado no runtime; não o atualize aqui
if not torch.cuda.is_available():
    raise RuntimeError('Selecione um runtime GPU, reinicie a sessão e execute de novo')
if any(entry.path.suffix.lower() in {'.dcm', '.dicom'} for entry in entries):
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pydicom>=3,<4',
                    'pylibjpeg>=2,<3', 'pylibjpeg-libjpeg>=2,<3', 'pylibjpeg-openjpeg>=2,<3'], check=True)
print('GPU:', torch.cuda.get_device_name())

In [ ]:
from datetime import datetime, timezone
from zipfile import ZIP_DEFLATED, ZipFile


def pack_results():
    """Compacta toda a saída; roda mesmo se o experimento falhar."""
    archive = OUTPUT / 'sprint-08-resultados.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as stream:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path != archive:
                stream.write(path, path.relative_to(OUTPUT))
    assert ZipFile(archive).testzip() is None, 'ZIP corrompido'
    return archive


try:
    from project_cnn_mammo.protocols import run_sprint
    out = OUTPUT / 'sprint-08' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    print(run_sprint(8, DATA, out, smoke=SMOKE, arm=ARM, manifest_path=globals().get('MANIFEST')))
    print('Saída:', out)
finally:
    archive = pack_results()
    print('ZIP:', archive, archive.stat().st_size, 'bytes')
    print('Baixe o ZIP em Save Version > Output ou na aba Data do notebook.')